# Dictionnaire de correspondance RPG vers CPS - SCOT RPB

Ce notebook construit le dictionnaire de correspondance entre les codes cultures du Registre Parcellaire Graphique (RPG) et les codes de Production Brute Standard (CPS/PBS) pour le territoire du SCOT Rhone Provence Baronnies.

**Sources officielles :**
- Reglement (CE) n 1242/2008 - Typologie communautaire des exploitations agricoles
- Agreste - Coefficients de Production Standard 2020
- IGN - Registre Parcellaire Graphique 2023

**Territoire :**
- SCOT RPB : 3 departements (Ardeche 07, Drome 26, Vaucluse 84)
- Ardeche + Drome : coefficients Rhone-Alpes
- Vaucluse : coefficients PACA

## 1. Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Chemins des fichiers
# Modifier ces chemins selon votre environnement

FICHIER_REF_RPG = "REF_CULTURES_GROUPES_CULTURES_2023.csv"
FICHIER_CPS_2020 = "Tableau_des_Coefficients_de_Production_Brute_Standard_en_2020.xlsx"
FEUILLE_CPS_SCOT = "cps_scot"

FICHIER_SORTIE_CSV = "DICTIONNAIRE_RPG_CPS_SCOT_RPB.csv"
FICHIER_SORTIE_XLSX = "DICTIONNAIRE_RPG_CPS_SCOT_RPB.xlsx"

In [3]:
# Codes cultures presents sur le territoire du SCOT RPB
# Extraits du fichier RPG_pac_RPB.gpkg

CODES_RPG_TERRAIN = [
    'AAR', 'AFG', 'AGR', 'AIL', 'AME', 'ARP', 'ART', 'AVH', 'AVP',
    'BDH', 'BDP', 'BFS', 'BOR', 'BTA', 'BTH', 'BTN', 'BTP', 'CAG',
    'CAH', 'CAR', 'CBT', 'CCN', 'CHU', 'CHV', 'CID', 'CML', 'CPL',
    'CTG', 'CZH', 'CZP', 'EPE', 'EPI', 'FEV', 'FLA', 'FLP', 'FNU',
    'FRA', 'FVL', 'GRA', 'HPC', 'JAC', 'LAV', 'LBF', 'LEC', 'LIP',
    'LOT', 'LUZ', 'MCR', 'MCS', 'MDI', 'MID', 'MIS', 'MLC', 'MLF',
    'MLG', 'MLO', 'MLT', 'MOH', 'MOT', 'MPC', 'NOS', 'NOX', 'NVT',
    'OAG', 'OHR', 'OIG', 'OLI', 'ORH', 'ORP', 'PAG', 'PCH', 'PEP',
    'PEV', 'PFR', 'PHF', 'PHI', 'PHS', 'PME', 'POR', 'POT', 'PPH',
    'PPP', 'PPR', 'PRF', 'PRU', 'PSL', 'PTC', 'PTR', 'PVP', 'PVT',
    'PWT', 'RDI', 'SAI', 'SGH', 'SOG', 'SOJ', 'SPH', 'SPL', 'SRS',
    'TBT', 'TCR', 'TOM', 'TRE', 'TRN', 'TRU', 'TTH', 'TTP', 'VES',
    'VRC', 'VRG'
]

## 2. Dictionnaire de correspondance RPG vers CPS

Ce dictionnaire peut etre modifie manuellement sans casser le reste du code.

Format : `'CODE_RPG': 'CODE_CPS'`

Source : Reglement (CE) n 1242/2008, Annexe I, Partie B

In [4]:
MAPPING_RPG_CPS = {
    
    # CEREALES
    # Groupe 1 - Ble tendre (Reglement 2.01.01.01)
    'BTH': 'C1110T',    # Ble tendre hiver
    'BTP': 'C1110T',    # Ble tendre printemps
    'EPE': 'C1110T',    # Epeautre
    
    # Ble dur (Reglement 2.01.01.02)
    'BDH': 'C1120T',    # Ble dur hiver
    'BDP': 'C1120T',    # Ble dur printemps
    
    # Orge (Reglement 2.01.01.04)
    'ORH': 'C1300T',    # Orge hiver
    'ORP': 'C1300T',    # Orge printemps
    
    # Avoine (Reglement 2.01.01.05)
    'AVH': 'C1400T',    # Avoine hiver
    'AVP': 'C1400T',    # Avoine printemps
    
    # Seigle (Reglement 2.01.01.03)
    'SGH': 'C1200T',    # Seigle
    
    # Autres cereales (Reglement 2.01.01.99)
    'TTH': 'C1600T_C1700T_C1900T',    # Triticale hiver
    'TTP': 'C1600T_C1700T_C1900T',    # Triticale printemps
    'SOG': 'C1600T_C1700T_C1900T',    # Sorgho
    'MLT': 'C1600T_C1700T_C1900T',    # Millet
    'SRS': 'C1600T_C1700T_C1900T',    # Sarrasin
    'CAG': 'C1600T_C1700T_C1900T',    # Autres cereales printemps
    'CAH': 'C1600T_C1700T_C1900T',    # Autres cereales hiver
    'MCR': 'C1600T_C1700T_C1900T',    # Melange cereales hiver
    'MCS': 'C1600T_C1700T_C1900T',    # Melange cereales printemps
    
    # Mais grain (Reglement 2.01.01.06)
    'MIS': 'C1500T',    # Mais grain
    'MID': 'C1500T',    # Mais doux
    
    # OLEAGINEUX
    # Colza (Reglement 2.01.06.04)
    'CZH': 'I1110T',    # Colza hiver
    'CZP': 'I1110T',    # Colza printemps
    
    # Tournesol (Reglement 2.01.06.05)
    'TRN': 'I1120T',    # Tournesol
    
    # Soja (Reglement 2.01.06.06)
    'SOJ': 'I1130T',    # Soja
    
    # Lin oleagineux (Reglement 2.01.06.07)
    'LIP': 'I1140T',    # Lin oleagineux
    
    # Autres oleagineux (Reglement 2.01.06.08)
    'CML': 'I1190T',    # Cameline
    'OAG': 'I1190T',    # Autres oleagineux printemps
    'OHR': 'I1190T',    # Autres oleagineux hiver
    'MOT': 'I1190T',    # Moutarde
    
    # PLANTES TEXTILES
    # Chanvre (Reglement 2.01.06.10)
    'CHV': 'I2200T',    # Chanvre
    
    # PLANTES AROMATIQUES ET PARFUM (Reglement 2.01.06.12)
    'LAV': 'I5000T',    # Lavande et lavandin
    'PRF': 'I5000T',    # Plantes a parfum perennes
    'AME': 'I5000T',    # Plantes medicinales annuelles
    'PME': 'I5000T',    # Plantes medicinales perennes
    'PPP': 'I5000T',    # Plantes medicinales arbustives
    'AAR': 'I5000T',    # Plantes aromatiques annuelles
    'ARP': 'I5000T',    # Plantes aromatiques perennes
    'PSL': 'I5000T',    # Persil
    'FNU': 'I5000T',    # Fenugrec
    
    # PLANTES SARCLEES
    # Betterave (Reglement 2.01.04)
    'BTN': 'R2000T',    # Betterave
    
    # PROTEAGINEUX (Reglement 2.01.02)
    'FEV': 'P1000T',    # Feve
    'FVL': 'P1000T',    # Feverole
    'PHI': 'P1000T',    # Pois proteagineux hiver
    'PPR': 'P1000T',    # Pois proteagineux printemps
    'MPC': 'P1000T',    # Melange proteagineux-cereales
    'PAG': 'P1000T',    # Autres legumineuses
    
    # Legumineuses a grains
    'LEC': 'P0000T',    # Lentille
    'PCH': 'P0000T',    # Pois chiche
    'PHF': 'P0000T',    # Pois/haricot frais
    'PHS': 'P0000T',    # Pois/haricot secs
    
    # FOURRAGES (Reglement 2.01.09)
    # Legumineuses fourrageres
    'LUZ': 'G2000T',    # Luzerne
    'TRE': 'G2000T',    # Trefle
    'SAI': 'G2000T',    # Sainfoin
    'LOT': 'G2000T',    # Lotier
    'VES': 'G2000T',    # Vesce
    'MLF': 'G2000T',    # Melange legumineuses fourrageres
    'MLC': 'G2000T',    # Melange legumineuses preponderantes
    'MLG': 'G2000T',    # Melange legumineuses-graminees
    
    # Autres fourrages
    'AFG': 'G9100T_G9900T',    # Autres fourrages annuels
    'CPL': 'G9100T_G9900T',    # Melange multi-especes
    'MOH': 'G9100T_G9900T',    # Moha
    
    # Graminees
    'GRA': 'G1000T',    # Graminees pures
    
    # PRAIRIES
    # Prairies temporaires (Reglement 2.01.09.01)
    'PTR': 'G1000T',    # Prairies temporaires
    
    # Prairies permanentes (Reglement 2.03.01)
    'PPH': 'J1000T',    # Prairies permanentes
    
    # Paturages pauvres (Reglement 2.03.02)
    'SPH': 'J2000T',    # Surfaces pastorales herbe
    'SPL': 'J2000T',    # Surfaces pastorales ligneuses
    
    # JACHERES ET BORDURES (Reglement 2.03.03)
    'JAC': 'J3000TE',   # Jachere
    'BOR': 'J3000TE',   # Bordure de champ
    'BFS': 'J3000TE',   # Bordure de foret
    'BTA': 'J3000TE',   # Bande tampon
    
    # LEGUMES ET MARAICHAGE (Reglement 2.01.07)
    # Pomme de terre (Reglement 2.01.03)
    'PTC': 'R1000T',    # Pomme de terre
    
    # Maraichage
    'MDI': 'V0000_S0000TK',    # Maraichage diversifie
    'TOM': 'V0000_S0000TK',    # Tomate
    'CAR': 'V0000_S0000TK',    # Carotte
    'AIL': 'V0000_S0000TK',    # Ail
    'OIG': 'V0000_S0000TK',    # Oignon
    'POR': 'V0000_S0000TK',    # Poireau
    'CHU': 'V0000_S0000TK',    # Chou
    'LBF': 'V0000_S0000TK',    # Laitue et salades
    'EPI': 'V0000_S0000TK',    # Epinard
    'CCN': 'V0000_S0000TK',    # Concombre et courgette
    'MLO': 'V0000_S0000TK',    # Melon
    'POT': 'V0000_S0000TK',    # Potiron et courges
    'PVP': 'V0000_S0000TK',    # Poivron et aubergine
    'ART': 'V0000_S0000TK',    # Artichaut
    'NVT': 'V0000_S0000TK',    # Navet et legumes racines
    'RDI': 'V0000_S0000TK',    # Radis
    'FLA': 'V0000_S0000TK',    # Autre legume annuel
    'FLP': 'V0000_S0000TK',    # Autre legume perenne
    'TBT': 'V0000_S0000TK',    # Tubercule tropical
    'FRA': 'V0000_S0000TK',    # Fraise
    
    # HORTICULTURE (Reglement 2.01.08)
    'HPC': 'N0000T',    # Horticulture ornementale
    
    # FRUITS (Reglement 2.04.01)
    # Fruits a noyaux
    'PVT': 'F1200T',    # Peche et nectarine
    'PRU': 'F1200T',    # Prune
    'CBT': 'F1200T',    # Cerise
    
    # Fruits a pepins
    'PWT': 'F1100T',    # Poire
    
    # Baies
    'PFR': 'F3000T',    # Petits fruits a baies
    
    # Agrumes
    'AGR': 'T0000T',    # Agrumes
    
    # Autres vergers
    'VRG': 'F0000T',    # Autres vergers
    
    # FRUITS A COQUE (Reglement 2.04.01.03)
    'NOX': 'F4000T',    # Noix
    'NOS': 'F4000T',    # Noisette
    'CTG': 'F4000T',    # Chataigne
    'TRU': 'F4000T',    # Truffiere
    
    # VIGNES (Reglement 2.04.04)
    'VRC': 'W1100T',    # Vigne
    
    # OLIVIERS (Reglement 2.04.03)
    'OLI': 'O1000T',    # Olive
    
    # PEPINIERES (Reglement 2.04.05)
    'PEP': 'L0000T',    # Pepiniere plus de 1 an
    'PEV': 'L0000T',    # Pepiniere moins de 1 an
    
    # AUTRES CULTURES
    'CID': 'ARA99T_ARA09S',    # Cultures inter-rangs
    'TCR': 'PECR9_H9000T',     # Taillis courte rotation
}

## 3. Seuils de classification IQE

Source : CGAAER (2020)

In [5]:
SEUILS_IQE = {
    5: (15000, float('inf')),   # >= 15 000 euros/ha
    4: (8000, 15000),           # 8 000 - 15 000 euros/ha
    3: (4000, 8000),            # 4 000 - 8 000 euros/ha
    2: (2000, 4000),            # 2 000 - 4 000 euros/ha
    1: (0, 2000),               # < 2 000 euros/ha
}

LIBELLES_IQE = {
    5: "Tres haute valeur economique",
    4: "Haute valeur economique",
    3: "Valeur economique moyenne",
    2: "Faible valeur economique",
    1: "Tres faible valeur economique",
}

## 4. Fonctions

In [6]:
def calcul_score_iqe(pbs):
    """Calcule le score IQE (1-5) a partir de la PBS en euros/ha."""
    if pd.isna(pbs) or pbs <= 0:
        return 1
    for score, (seuil_min, seuil_max) in SEUILS_IQE.items():
        if seuil_min <= pbs < seuil_max:
            return score
    return 1


def charger_ref_rpg(chemin):
    """Charge le referentiel RPG des cultures."""
    df = pd.read_csv(chemin, sep=';', encoding='latin-1')
    df['CODE_CULTURE'] = df['CODE_CULTURE'].str.strip().str.upper()
    df['CODE_GROUPE_CULTURE'] = pd.to_numeric(df['CODE_GROUPE_CULTURE'], errors='coerce').fillna(0).astype(int)
    df['LIBELLE_CULTURE'] = df['LIBELLE_CULTURE'].str.strip()
    df['LIBELLE_GROUPE_CULTURE'] = df['LIBELLE_GROUPE_CULTURE'].str.strip()
    return df


def charger_cps_scot(chemin, feuille):
    """Charge la feuille CPS SCOT avec les coefficients des deux regions."""
    df = pd.read_excel(chemin, sheet_name=feuille)
    df.columns = ['CODE_CPS', 'LIBELLE_CPS', 'PBS_PACA', 'PBS_RHONE_ALPES']
    df['CODE_CPS'] = df['CODE_CPS'].str.strip()
    
    # Conversion champignons de euros/100m2 en euros/ha
    masque = df['CODE_CPS'] == 'U1000'
    if masque.any():
        df.loc[masque, 'PBS_PACA'] = df.loc[masque, 'PBS_PACA'] * 100
        df.loc[masque, 'PBS_RHONE_ALPES'] = df.loc[masque, 'PBS_RHONE_ALPES'] * 100
        print("Note : Champignons (U1000) convertis de euros/100m2 en euros/ha")
    
    return df


def construire_dictionnaire(codes_terrain, ref_rpg, cps_scot, mapping):
    """Construit le dictionnaire de correspondance complet."""
    resultats = []
    codes_non_mappes = []
    
    for code_rpg in codes_terrain:
        info_rpg = ref_rpg[ref_rpg['CODE_CULTURE'] == code_rpg]
        
        if len(info_rpg) > 0:
            libelle_rpg = info_rpg.iloc[0]['LIBELLE_CULTURE']
            code_groupe = info_rpg.iloc[0]['CODE_GROUPE_CULTURE']
            libelle_groupe = info_rpg.iloc[0]['LIBELLE_GROUPE_CULTURE']
        else:
            libelle_rpg = "NON TROUVE"
            code_groupe = 0
            libelle_groupe = "INCONNU"
        
        if code_rpg in mapping:
            code_cps = mapping[code_rpg]
            info_cps = cps_scot[cps_scot['CODE_CPS'] == code_cps]
            
            if len(info_cps) > 0:
                libelle_cps = info_cps.iloc[0]['LIBELLE_CPS']
                pbs_paca = info_cps.iloc[0]['PBS_PACA']
                pbs_ra = info_cps.iloc[0]['PBS_RHONE_ALPES']
            else:
                libelle_cps = "CODE CPS NON TROUVE"
                pbs_paca = 0
                pbs_ra = 0
        else:
            code_cps = ""
            libelle_cps = ""
            pbs_paca = 0
            pbs_ra = 0
            codes_non_mappes.append(code_rpg)
        
        score_paca = calcul_score_iqe(pbs_paca)
        score_ra = calcul_score_iqe(pbs_ra)
        
        resultats.append({
            'CODE_RPG': code_rpg,
            'LIBELLE_RPG': libelle_rpg,
            'CODE_GROUPE_RPG': code_groupe,
            'LIBELLE_GROUPE_RPG': libelle_groupe,
            'CODE_CPS': code_cps,
            'LIBELLE_CPS': libelle_cps,
            'PBS_RHONE_ALPES': round(pbs_ra, 2) if pd.notna(pbs_ra) else 0.0,
            'PBS_PACA': round(pbs_paca, 2) if pd.notna(pbs_paca) else 0.0,
            'SCORE_IQE_RHONE_ALPES': score_ra,
            'SCORE_IQE_PACA': score_paca,
            'NIVEAU_IQE_RHONE_ALPES': LIBELLES_IQE[score_ra],
            'NIVEAU_IQE_PACA': LIBELLES_IQE[score_paca],
        })
    
    df = pd.DataFrame(resultats)
    df = df.sort_values('PBS_RHONE_ALPES', ascending=False).reset_index(drop=True)
    
    return df, codes_non_mappes

## 5. Chargement des donnees

In [8]:
ref_rpg = charger_ref_rpg(FICHIER_REF_RPG)
print(f"Referentiel RPG : {len(ref_rpg)} codes cultures")

cps_scot = charger_cps_scot(FICHIER_CPS_2020, FEUILLE_CPS_SCOT)
print(f"CPS SCOT : {len(cps_scot)} coefficients")

Referentiel RPG : 372 codes cultures
Note : Champignons (U1000) convertis de euros/100m2 en euros/ha
CPS SCOT : 68 coefficients


In [9]:
ref_terrain = ref_rpg[ref_rpg['CODE_CULTURE'].isin(CODES_RPG_TERRAIN)]
print(f"Codes presents sur le terrain : {len(CODES_RPG_TERRAIN)}")
print(f"Codes trouves dans le referentiel : {len(ref_terrain)}")

Codes presents sur le terrain : 110
Codes trouves dans le referentiel : 110


## 6. Construction du dictionnaire

In [10]:
dictionnaire, non_mappes = construire_dictionnaire(
    CODES_RPG_TERRAIN, 
    ref_rpg, 
    cps_scot, 
    MAPPING_RPG_CPS
)

print(f"Codes mappes : {len(dictionnaire) - len(non_mappes)}")
print(f"Codes non mappes : {len(non_mappes)}")

if non_mappes:
    print("\nCodes non mappes :")
    for code in non_mappes:
        info = ref_rpg[ref_rpg['CODE_CULTURE'] == code]
        if len(info) > 0:
            print(f"  {code}: {info.iloc[0]['LIBELLE_CULTURE']}")

Codes mappes : 110
Codes non mappes : 0


In [11]:
dictionnaire.head(20)

,CODE_RPG,LIBELLE_RPG,CODE_GROUPE_RPG,LIBELLE_GROUPE_RPG,CODE_CPS,LIBELLE_CPS,PBS_RHONE_ALPES,PBS_PACA,SCORE_IQE_RHONE_ALPES,SCORE_IQE_PACA,NIVEAU_IQE_RHONE_ALPES,NIVEAU_IQE_PACA
0,HPC,Horticulture ornementale,25,Légumes ou fleurs,N0000T,Fleurs et plantes ornementales (non compris pé...,112694.40,112694.40,5,5,Tres haute valeur economique,Tres haute valeur economique
1,PEP,Pépinière (plants laissés en terre plus dun an),28,Divers,L0000T,Pépinières,30952.00,30952.00,5,5,Tres haute valeur economique,Tres haute valeur economique
2,PEV,Pépinière (plants laissés en terre moins dun an),28,Divers,L0000T,Pépinières,30952.00,30952.00,5,5,Tres haute valeur economique,Tres haute valeur economique
3,CHU,Chou,25,Légumes ou fleurs,V0000_S0000TK,"Légumes frais, melons, fraises, culture maraîc...",28966.08,40324.23,5,5,Tres haute valeur economique,Tres haute valeur economique
4,CAR,Carotte,25,Légumes ou fleurs,V0000_S0000TK,"Légumes frais, melons, fraises, culture maraîc...",28966.08,40324.23,5,5,Tres haute valeur economique,Tres haute valeur economique
5,AIL,Aïl,25,Légumes ou fleurs,V0000_S0000TK,"Légumes frais, melons, fraises, culture maraîc...",28966.08,40324.23,5,5,Tres haute valeur economique,Tres haute valeur economique
6,LBF,"Laitue, endive et autres salades",25,Légumes ou fleurs,V0000_S0000TK,"Légumes frais, melons, fraises, culture maraîc...",28966.08,40324.23,5,5,Tres haute valeur economique,Tres haute valeur economique
7,EPI,"Epinard, oseille et bette",25,Légumes ou fleurs,V0000_S0000TK,"Légumes frais, melons, fraises, culture maraîc...",28966.08,40324.23,5,5,Tres haute valeur economique,Tres haute valeur economique
8,FRA,Fraise (en pleine terre),25,Légumes ou fleurs,V0000_S0000TK,"Légumes frais, melons, fraises, culture maraîc...",28966.08,40324.23,5,5,Tres haute valeur economique,Tres haute valeur economique
9,FLA,Autre légume ou fruit annuel,25,Légumes ou fleurs,V0000_S0000TK,"Légumes frais, melons, fraises, culture maraîc...",28966.08,40324.23,5,5,Tres haute valeur economique,Tres haute valeur economique


## 7. Statistiques

In [12]:
print("Distribution des scores IQE (Rhone-Alpes)")
print("-" * 50)
for score in [5, 4, 3, 2, 1]:
    n = len(dictionnaire[dictionnaire['SCORE_IQE_RHONE_ALPES'] == score])
    pct = n / len(dictionnaire) * 100
    print(f"Score {score} : {n:3} cultures ({pct:5.1f}%) - {LIBELLES_IQE[score]}")

print("\nDistribution des scores IQE (PACA)")
print("-" * 50)
for score in [5, 4, 3, 2, 1]:
    n = len(dictionnaire[dictionnaire['SCORE_IQE_PACA'] == score])
    pct = n / len(dictionnaire) * 100
    print(f"Score {score} : {n:3} cultures ({pct:5.1f}%) - {LIBELLES_IQE[score]}")

Distribution des scores IQE (Rhone-Alpes)
--------------------------------------------------
Score 5 :  30 cultures ( 27.3%) - Tres haute valeur economique
Score 4 :   3 cultures (  2.7%) - Haute valeur economique
Score 3 :   6 cultures (  5.5%) - Valeur economique moyenne
Score 2 :  10 cultures (  9.1%) - Faible valeur economique
Score 1 :  61 cultures ( 55.5%) - Tres faible valeur economique

Distribution des scores IQE (PACA)
--------------------------------------------------
Score 5 :  32 cultures ( 29.1%) - Tres haute valeur economique
Score 4 :   1 cultures (  0.9%) - Haute valeur economique
Score 3 :   1 cultures (  0.9%) - Valeur economique moyenne
Score 2 :  15 cultures ( 13.6%) - Faible valeur economique
Score 1 :  61 cultures ( 55.5%) - Tres faible valeur economique


In [13]:
print("PBS moyennes")
print("-" * 30)
print(f"Rhone-Alpes : {dictionnaire['PBS_RHONE_ALPES'].mean():,.2f} euros/ha")
print(f"PACA        : {dictionnaire['PBS_PACA'].mean():,.2f} euros/ha")

PBS moyennes
------------------------------
Rhone-Alpes : 9,441.34 euros/ha
PACA        : 12,158.99 euros/ha


In [14]:
print("Top 10 cultures (PBS Rhone-Alpes)")
print("-" * 70)
top10 = dictionnaire.nlargest(10, 'PBS_RHONE_ALPES')[['CODE_RPG', 'LIBELLE_RPG', 'PBS_RHONE_ALPES', 'SCORE_IQE_RHONE_ALPES']]
for _, row in top10.iterrows():
    print(f"{row['CODE_RPG']:5} {row['LIBELLE_RPG'][:40]:<40} {row['PBS_RHONE_ALPES']:>12,.2f} euros/ha  Score {row['SCORE_IQE_RHONE_ALPES']}")

Top 10 cultures (PBS Rhone-Alpes)
----------------------------------------------------------------------
HPC   Horticulture ornementale                   112,694.40 euros/ha  Score 5
PEP   Pépinière (plants laissés en terre plus     30,952.00 euros/ha  Score 5
PEV   Pépinière (plants laissés en terre moins    30,952.00 euros/ha  Score 5
CHU   Chou                                        28,966.08 euros/ha  Score 5
CAR   Carotte                                     28,966.08 euros/ha  Score 5
AIL   Aïl                                         28,966.08 euros/ha  Score 5
LBF   Laitue, endive et autres salades            28,966.08 euros/ha  Score 5
EPI   Epinard, oseille et bette                   28,966.08 euros/ha  Score 5
FRA   Fraise (en pleine terre)                    28,966.08 euros/ha  Score 5
FLA   Autre légume ou fruit annuel                28,966.08 euros/ha  Score 5


## 8. Export

In [15]:
dictionnaire.to_csv(FICHIER_SORTIE_CSV, index=False, sep=';', encoding='utf-8-sig')
print(f"Export CSV : {FICHIER_SORTIE_CSV}")

Export CSV : DICTIONNAIRE_RPG_CPS_SCOT_RPB.csv


In [16]:
dictionnaire.to_excel(FICHIER_SORTIE_XLSX, index=False, sheet_name='Dictionnaire')
print(f"Export Excel : {FICHIER_SORTIE_XLSX}")

Export Excel : DICTIONNAIRE_RPG_CPS_SCOT_RPB.xlsx


In [17]:
print("Traitement termine")
print(f"Fichiers generes : {FICHIER_SORTIE_CSV}, {FICHIER_SORTIE_XLSX}")

Traitement termine
Fichiers generes : DICTIONNAIRE_RPG_CPS_SCOT_RPB.csv, DICTIONNAIRE_RPG_CPS_SCOT_RPB.xlsx
